# SCH-LightNet: Full Working Reference Implementation
**Multimodal spectral–temporal learning and uncertainty-aware progression prediction for paddy sheath blight**

This Colab-ready notebook implements a runnable research prototype inspired by the uploaded SCH-LightNet manuscript:
- RGB paddy images with resizing and conditional CLAHE
- Six environmental channels and irregular visit times
- Hampel-style sensor filtering, robust scaling, time/frequency descriptors
- Coupling-guided low-rank interaction embedding (CGCI-inspired)
- MobileNetV3 image encoder + temporal convolution network + low-rank bilinear fusion + hypergraph-style fusion
- Low-rank continuous-time-inspired state evolution
- Multi-horizon risk, ordinal severity, discrete-time hazard outputs
- Split-conformal prediction sets, metrics, model/checkpoint and CSV exports

## Important reproducibility note
The manuscript states that no public repository jointly contains images, microclimate, metadata, and onset dates, and describes simulator-assisted semi-synthetic episode assembly. Therefore, the notebook **does not pretend that generated example labels are real field observations**. If you provide your own aligned episode manifest/sensor records, it trains on them. If no data are found, it creates a small synthetic smoke-test dataset so the notebook runs end-to-end; results from that fallback are only for testing the code, not scientific claims.

The paper lists image repositories and sensor repositories in Section 3.2. Download them according to their terms and point `DATA_ROOT` below at your local/Drive copy. A recommended image folder layout is `images/<severity_name>/*.jpg`. For aligned real episodes, use the manifest format described in the data-loading cell.

## 1. Install dependencies

In [ ]:
# Run in Google Colab. Restart runtime only if Colab requests it.
!pip -q install timm PyWavelets scikit-learn pandas numpy pillow opencv-python-headless matplotlib seaborn tqdm openpyxl

## 2. Imports, reproducibility, and configuration

In [ ]:
import os, json, math, random, time, glob, warnings, zipfile, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, brier_score_loss,
                             log_loss, confusion_matrix, classification_report,
                             cohen_kappa_score)
from sklearn.impute import SimpleImputer

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Change this to your Google Drive dataset folder if using Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

DATA_ROOT = Path('/content/drive/MyDrive/SCH_LightNet_Dataset')
OUTPUT_DIR = Path('/content/drive/MyDrive/SCH_LightNet_Outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
SENSOR_CHANNELS = ['air_temperature','relative_humidity','rainfall',
                   'soil_moisture','leaf_wetness','wind_speed']
SENSOR_WINDOW = 96
NUM_SEVERITY_CLASSES = 6
HORIZONS = [7, 14, 21, 30]
BATCH_SIZE = 8
EPOCHS = 5          # increase to 50-100 for real experiments
LR = 2e-4
NUM_WORKERS = 2
USE_PRETRAINED = True
print("Data root:", DATA_ROOT, "| outputs:", OUTPUT_DIR)

## 3. Data format and automatic discovery

### Option A — image-only folders (for a smoke test / image supervision)
Place images in `DATA_ROOT/images/<class_name>/*.jpg`. Severity folder names should correspond to six ordered grades, for example `0`, `1`, `2`, `3`, `4`, `5` or named folders. This supplies images and severity labels, but does **not** supply real progression/onset labels.

### Option B — aligned episode manifest (preferred)
Create `episodes.csv` with one row per visit and at least these columns:
- `episode_id`, `visit_day`, `image_path`, `severity`
- the six sensor columns listed above
- optional `cultivar`, `growth_stage`, `day_of_year`
- episode-level onset columns: `onset_day` (days from episode start; blank if not observed) and `event_observed` (1 if onset observed, 0 if right-censored)

Rows with the same `episode_id` form an irregularly sampled episode. The example code below aggregates each episode into one training sample using the latest visit, while retaining the last 96 sensor observations. For a full longitudinal CDE experiment, keep the entire visit sequence and adapt the episode dataset accordingly.

In [ ]:
MANIFEST_PATH = DATA_ROOT / 'episodes.csv'
IMAGE_DIR = DATA_ROOT / 'images'

def discover_image_folders(root):
    found = []
    if root.exists():
        for p in root.iterdir():
            if p.is_dir():
                imgs = [x for x in p.rglob('*') if x.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}]
                if imgs: found.append((p.name, len(imgs), p))
    return found

print("Image folders:", discover_image_folders(IMAGE_DIR)[:20])
print("Manifest exists:", MANIFEST_PATH.exists())

## 4. Preprocessing utilities

In [ ]:
def hampel_filter_1d(x, half_window=6, n_sigmas=3.0):
    """Hampel-style robust outlier replacement; x is a 1-D array."""
    x = np.asarray(x, dtype=float).copy()
    if len(x) < 3: return x
    out = x.copy()
    for i in range(len(x)):
        lo, hi = max(0, i-half_window), min(len(x), i+half_window+1)
        window = x[lo:hi]
        med = np.nanmedian(window)
        mad = np.nanmedian(np.abs(window-med))
        sigma = 1.4826 * mad
        if np.isfinite(x[i]) and np.isfinite(med) and sigma > 1e-8 and abs(x[i]-med) > n_sigmas*sigma:
            out[i] = med
    return out

def sensor_descriptors(sensor_array):
    """Compact descriptors: mean/std/min/max/slope/energy + frequency-band summaries."""
    a = np.asarray(sensor_array, dtype=np.float32)
    if a.ndim == 1: a = a[:, None]
    feats = []
    for c in range(a.shape[1]):
        x = np.nan_to_num(a[:, c], nan=0.0)
        feats.extend([x.mean(), x.std()+1e-6, x.min(), x.max(),
                      np.polyfit(np.arange(len(x)), x, 1)[0] if len(x)>1 else 0.0,
                      np.mean(x*x)])
        # FFT summaries are used as a practical time-frequency approximation.
        spec = np.abs(np.fft.rfft(x))
        if len(spec) > 2:
            feats.extend([spec[1:max(2,len(spec)//4)].mean(),
                          spec[max(1,len(spec)//4):].mean()])
        else:
            feats.extend([0.0, 0.0])
    return np.asarray(feats, dtype=np.float32)

def conditional_clahe_rgb(pil_img):
    rgb = np.array(pil_img.convert('RGB'))
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    if 60 <= float(l.mean()) <= 200:
        # As in the manuscript, apply CLAHE only outside the acceptable luminance range.
        return Image.fromarray(rgb)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    return Image.fromarray(cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2RGB))

def safe_image(path):
    try:
        return conditional_clahe_rgb(Image.open(path).convert('RGB')).resize((IMG_SIZE, IMG_SIZE))
    except Exception:
        return Image.new('RGB', (IMG_SIZE, IMG_SIZE), color=(120,120,120))

## 5. Load data (real manifest when available; synthetic smoke test otherwise)

In [ ]:
def load_manifest(manifest_path):
    df = pd.read_csv(manifest_path)
    required = {'episode_id','visit_day','image_path','severity'}
    missing = required - set(df.columns)
    if missing: raise ValueError(f"episodes.csv missing required columns: {sorted(missing)}")
    for c in SENSOR_CHANNELS:
        if c not in df.columns: df[c] = np.nan
    if 'event_observed' not in df.columns: df['event_observed'] = 0
    if 'onset_day' not in df.columns: df['onset_day'] = np.nan
    if 'cultivar' not in df.columns: df['cultivar'] = 'unknown'
    if 'growth_stage' not in df.columns: df['growth_stage'] = 0
    if 'day_of_year' not in df.columns: df['day_of_year'] = 1
    df['image_path'] = df['image_path'].apply(lambda p: str((DATA_ROOT / p).resolve()) if not os.path.isabs(str(p)) else str(p))
    return df

def make_synthetic_data(n=120):
    """Deterministic synthetic data solely to verify all code paths execute."""
    rows=[]
    synth_dir = OUTPUT_DIR / 'synthetic_images'
    synth_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(SEED)
    for i in range(n):
        sev = int(rng.integers(0, NUM_SEVERITY_CLASSES))
        # Make a simple generated leaf-like image with colored lesion-like spots.
        arr = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        arr[:] = [40, 100 + int(8*sev), 35]
        yy, xx = np.ogrid[:IMG_SIZE,:IMG_SIZE]
        for _ in range(5 + sev*4):
            cx, cy = int(rng.integers(15, IMG_SIZE-15)), int(rng.integers(15, IMG_SIZE-15))
            r = int(rng.integers(3, 10))
            mask = (xx-cx)**2 + (yy-cy)**2 <= r*r
            arr[mask] = [120+sev*15, 90, 35]
        ip = synth_dir / f'episode_{i:04d}.png'
        Image.fromarray(arr).save(ip)
        row={'episode_id':f'syn_{i:04d}','visit_day':int(rng.integers(0,30)),
             'image_path':str(ip),'severity':sev,'cultivar':'unknown',
             'growth_stage':int(rng.integers(0,6)),'day_of_year':int(rng.integers(1,366)),
             'event_observed':int(rng.random()<0.65),
             'onset_day':float(rng.integers(1,31)) if rng.random()<0.65 else np.nan}
        vals = [28,82,2,35,60,1.5]
        for c,v in zip(SENSOR_CHANNELS, vals): row[c] = float(v + rng.normal(0, max(abs(v)*.08,.1)))
        rows.append(row)
    return pd.DataFrame(rows)

if MANIFEST_PATH.exists():
    raw_df = load_manifest(MANIFEST_PATH)
    DATA_MODE = 'manifest'
    print("Loaded aligned manifest:", raw_df.shape)
else:
    folder_info = discover_image_folders(IMAGE_DIR)
    if folder_info:
        # Use supplied class-folder images; environmental/onset targets below are explicitly
        # placeholders unless the user supplies an aligned episodes.csv.
        names = [x[0] for x in folder_info]
        def class_order(name):
            try: return int(name)
            except Exception: return None
        numeric = all(class_order(n) is not None for n in names)
        sorted_names = sorted(names, key=lambda n: int(n) if numeric else n.lower())
        class_map = {name:i for i,name in enumerate(sorted_names[:NUM_SEVERITY_CLASSES])}
        rng = np.random.default_rng(SEED)
        rows = []
        for folder_name, _, folder in folder_info:
            if folder_name not in class_map: continue
            for ip in folder.rglob('*'):
                if ip.suffix.lower() not in {'.jpg','.jpeg','.png','.bmp','.webp'}: continue
                row={'episode_id':f'img_{len(rows):06d}','visit_day':0,'image_path':str(ip),
                     'severity':class_map[folder_name],'cultivar':'unknown','growth_stage':0,
                     'day_of_year':1,'event_observed':int(rng.random()<0.5),
                     'onset_day':float(rng.integers(1,31)) if rng.random()<0.5 else np.nan}
                for c,v in zip(SENSOR_CHANNELS,[28,82,2,35,60,1.5]):
                    row[c]=float(v+rng.normal(0,max(abs(v)*.08,.1)))
                rows.append(row)
        raw_df=pd.DataFrame(rows)
        DATA_MODE='image_folders_with_synthetic_auxiliary_targets'
        print("Loaded class-folder images. NOTE: sensor/onset targets are synthetic placeholders; use episodes.csv for real multimodal training.")
    else:
        raw_df = make_synthetic_data()
        DATA_MODE = 'synthetic_smoke_test'
        print("No episodes.csv or image folders found. Running synthetic smoke test:", raw_df.shape)

raw_df.head()

## 6. Build episode-level multimodal samples

In [ ]:
# Each episode is represented by its most recent visit plus its preceding sensor records.
# The split is by episode_id to prevent visits from the same episode crossing partitions.
def encode_metadata(row):
    stage = int(np.clip(pd.to_numeric(row.get('growth_stage',0), errors='coerce') if pd.notna(row.get('growth_stage',0)) else 0, 0, 5))
    stage_code = np.zeros(6, dtype=np.float32)
    stage_code[:stage+1] = 1.0  # cumulative/thermometer code
    doy = float(row.get('day_of_year', 1) if pd.notna(row.get('day_of_year',1)) else 1)
    cyc = [math.sin(2*math.pi*doy/365.25), math.cos(2*math.pi*doy/365.25)]
    cultivar = str(row.get('cultivar','unknown'))
    # Stable compact cultivar hash; unknown is handled explicitly.
    cultivar_code = np.zeros(8, dtype=np.float32)
    cultivar_code[abs(hash(cultivar)) % 8] = 1.0
    return np.concatenate([stage_code, np.asarray(cyc,dtype=np.float32), cultivar_code])

def build_samples(df):
    samples=[]
    for eid, g in df.groupby('episode_id'):
        g = g.sort_values('visit_day')
        last = g.iloc[-1]
        # Clean each channel across the episode, interpolate short gaps, then pad/crop to 96.
        seq = g[SENSOR_CHANNELS].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
        for j in range(seq.shape[1]):
            seq[:,j] = hampel_filter_1d(seq[:,j], half_window=6, n_sigmas=3)
            s = pd.Series(seq[:,j]).interpolate(limit=2, limit_direction='both')
            seq[:,j] = s.fillna(np.nanmedian(seq[:,j]) if np.isfinite(np.nanmedian(seq[:,j])) else 0).to_numpy()
        if len(seq) < SENSOR_WINDOW:
            pad = np.repeat(seq[:1], SENSOR_WINDOW-len(seq), axis=0)
            seq = np.concatenate([pad,seq],axis=0)
        else: seq = seq[-SENSOR_WINDOW:]
        sev = int(np.clip(pd.to_numeric(last['severity'], errors='coerce'),0,NUM_SEVERITY_CLASSES-1))
        onset = pd.to_numeric(last.get('onset_day',np.nan), errors='coerce')
        observed_value = pd.to_numeric(last.get('event_observed',0), errors='coerce')
        observed = int(observed_value) if pd.notna(observed_value) else 0
        if pd.isna(onset): onset = 31.0
        # Multi-horizon labels; only forward onset if event observed.
        risk = [int(observed == 1 and float(onset) <= h) for h in HORIZONS]
        # Four interval labels for onset hazard; class 4 means no onset observed within 30 days.
        if observed == 0 or float(onset) > 30: onset_class = 4
        elif float(onset) <= 7: onset_class = 0
        elif float(onset) <= 14: onset_class = 1
        elif float(onset) <= 21: onset_class = 2
        else: onset_class = 3
        samples.append({
            'episode_id': str(eid), 'image_path': str(last['image_path']),
            'sensor': seq.astype(np.float32), 'sensor_desc': sensor_descriptors(seq),
            'metadata': encode_metadata(last), 'severity': sev, 'risk': np.array(risk,dtype=np.float32),
            'onset_class': onset_class, 'event_observed': observed,
            'onset_day': float(onset), 'visit_day': float(last['visit_day'])
        })
    return samples

samples = build_samples(raw_df)
print("Episode samples:", len(samples), "sensor shape:", samples[0]['sensor'].shape,
      "metadata:", samples[0]['metadata'].shape)

## 7. Split by episode and fit sensor scaler on training only

In [ ]:
indices = np.arange(len(samples))
severity_labels = np.array([s['severity'] for s in samples])
# Stratify where possible; fall back to random split if a class has too few examples.
try:
    train_idx, temp_idx = train_test_split(indices, test_size=0.30, random_state=SEED, stratify=severity_labels)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=SEED,
                                         stratify=severity_labels[temp_idx])
except ValueError:
    train_idx, temp_idx = train_test_split(indices, test_size=0.30, random_state=SEED)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=SEED)

train_rows = [samples[i] for i in train_idx]
val_rows = [samples[i] for i in val_idx]
test_rows = [samples[i] for i in test_idx]
calib_rows = val_rows  # reserved calibration partition; do not use it for gradient updates
print(f"Train={len(train_rows)}, calibration/validation={len(val_rows)}, test={len(test_rows)}")

# Fit robust scaler only on training sensor records.
scaler = RobustScaler()
scaler.fit(np.concatenate([s['sensor'] for s in train_rows], axis=0))
for group in (train_rows, val_rows, test_rows):
    for s in group:
        s['sensor_scaled'] = scaler.transform(s['sensor']).astype(np.float32)

## 8. Dataset and DataLoaders

In [ ]:
IMAGE_TRANSFORM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

class PaddyEpisodeDataset(Dataset):
    def __init__(self, rows, augment=False):
        self.rows = rows
        self.augment = augment
        self.aug = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08)
        ])
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        s = self.rows[i]
        img = safe_image(s['image_path'])
        if self.augment: img = self.aug(img)
        x_img = IMAGE_TRANSFORM(img)
        return {
            'image': x_img,
            'sensor': torch.tensor(s['sensor_scaled'].T, dtype=torch.float32),
            'sensor_desc': torch.tensor(s['sensor_desc'], dtype=torch.float32),
            'metadata': torch.tensor(s['metadata'], dtype=torch.float32),
            'severity': torch.tensor(s['severity'], dtype=torch.long),
            'risk': torch.tensor(s['risk'], dtype=torch.float32),
            'onset_class': torch.tensor(s['onset_class'], dtype=torch.long),
            'event_observed': torch.tensor(s['event_observed'], dtype=torch.float32),
            'onset_day': torch.tensor(s['onset_day'], dtype=torch.float32),
            'visit_day': torch.tensor(s['visit_day'], dtype=torch.float32),
            'episode_id': s['episode_id']
        }

train_loader = DataLoader(PaddyEpisodeDataset(train_rows, augment=True), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(PaddyEpisodeDataset(val_rows), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(PaddyEpisodeDataset(test_rows), batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

## 9. SCH-LightNet model (practical differentiable reference implementation)

In [ ]:
class CGCIInspired(nn.Module):
    """Coupling-guided low-rank interaction block; compact approximation of CGCI."""
    def __init__(self, channels=6, meta_dim=16, out_dim=64, rank=8):
        super().__init__()
        self.lag_logits = nn.Parameter(torch.zeros(8))
        self.sensor_proj = nn.Linear(channels*4, out_dim)
        self.meta_proj = nn.Linear(meta_dim, out_dim)
        self.rank_a = nn.Linear(out_dim, rank, bias=False)
        self.rank_b = nn.Linear(out_dim, rank, bias=False)
        self.rank_c = nn.Linear(out_dim, rank, bias=False)
        self.out = nn.Sequential(nn.Linear(rank, out_dim), nn.LayerNorm(out_dim), nn.GELU())
    def forward(self, x, desc, meta):
        # x: B,C,T. Use last 8 lag summaries with learned softmax gates.
        b,c,t = x.shape
        lag_weights = torch.softmax(self.lag_logits, dim=0)
        lag_feats=[]
        for lag in range(8):
            end = max(1, t-lag*max(1,t//8))
            start = max(0, end-max(1,t//8))
            lag_feats.append(x[:,:,start:end].mean(-1))
        lag_stack = torch.stack(lag_feats, dim=1)  # B,8,C
        gated = (lag_stack * lag_weights.view(1,8,1)).sum(1)
        # spectral descriptors add a compact frequency/complexity summary
        desc_summary = desc
        if desc_summary.shape[1] < 24:
            desc_summary = F.pad(desc_summary, (0,24-desc_summary.shape[1]))
        desc_summary = desc_summary[:,:24]
        raw = torch.cat([gated, desc_summary], dim=1)
        if raw.shape[1] < 6*4: raw = F.pad(raw,(0,6*4-raw.shape[1]))
        sensor = self.sensor_proj(raw[:,:24])
        context = self.meta_proj(meta)
        # CP-style factor contraction, never forms a full outer-product tensor.
        interaction = self.rank_a(sensor) * self.rank_b(context) * self.rank_c(sensor + context)
        return self.out(interaction)

class TemporalConvNet(nn.Module):
    def __init__(self, in_ch=6, channels=32, out_dim=64):
        super().__init__()
        blocks=[]
        c=in_ch
        for d in [1,2,4,8]:
            blocks += [nn.Conv1d(c, channels, kernel_size=3, padding=d, dilation=d),
                       nn.GELU(), nn.BatchNorm1d(channels), nn.Dropout(0.1)]
            c=channels
        self.net=nn.Sequential(*blocks)
        self.proj=nn.Linear(channels,out_dim)
    def forward(self,x):
        z=self.net(x)
        return self.proj(z.mean(-1))

class LowRankBilinear(nn.Module):
    def __init__(self, dim=64, rank=64, out_dim=64):
        super().__init__()
        self.v=nn.Linear(dim,rank,bias=False)
        self.t=nn.Linear(dim,rank,bias=False)
        self.out=nn.Linear(rank,out_dim)
    def forward(self,a,b): return self.out(self.v(a)*self.t(b))

class HypergraphFusion(nn.Module):
    """Small hypergraph-style message passing over image, temporal, CGCI, metadata nodes."""
    def __init__(self, dim=64):
        super().__init__()
        self.node_proj=nn.Linear(dim,dim)
        self.hyperedge=nn.Parameter(torch.randn(4,dim)*0.02)
        self.gate=nn.Sequential(nn.Linear(dim*4,dim),nn.GELU(),nn.Linear(dim,dim),nn.Sigmoid())
        self.norm=nn.LayerNorm(dim)
    def forward(self, nodes):
        # nodes: B,4,D. Each learned hyperedge receives weighted messages from all modalities.
        edge_w=torch.softmax(torch.einsum('bnd,ed->bne',nodes,self.hyperedge),dim=1)
        edge=torch.einsum('bne,bnd->bed',edge_w,nodes)
        pooled=edge.mean(1)
        cat=nodes.flatten(1)
        return self.norm(pooled + self.gate(cat)*self.node_proj(pooled))

class LowRankContinuousState(nn.Module):
    """Low-rank continuous-time-inspired state update over elapsed visit-day summary."""
    def __init__(self, input_dim=64, hidden=32, rank=8):
        super().__init__()
        self.init=nn.Linear(input_dim,hidden)
        self.left=nn.Linear(hidden,rank,bias=False)
        self.right=nn.Linear(rank,hidden,bias=False)
        self.gate=nn.Linear(input_dim,hidden)
        self.refine=nn.Sequential(nn.Linear(hidden,hidden*2),nn.GLU(),nn.LayerNorm(hidden))
    def forward(self,z,delta_days=None):
        h=torch.tanh(self.init(z))
        # Stable explicit midpoint-like low-rank integration with bounded steps.
        dt=torch.ones((z.size(0),1),device=z.device) if delta_days is None else delta_days.view(-1,1).to(z.device).clamp(0,30)/30
        k1=self.right(torch.tanh(self.left(h))) + 0.1*torch.tanh(self.gate(z))
        mid=h+0.5*dt*k1
        k2=self.right(torch.tanh(self.left(mid))) + 0.1*torch.tanh(self.gate(z))
        h=h+dt*k2
        return self.refine(h), k2

class SCHLightNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = models.MobileNet_V3_Large_Weights.DEFAULT if pretrained else None
        try:
            backbone=models.mobilenet_v3_large(weights=weights, width_mult=0.5)
        except Exception as e:
            print('Pretrained backbone unavailable; using random initialization:', str(e)[:160])
            backbone=models.mobilenet_v3_large(weights=None, width_mult=0.5)
        self.image_backbone=backbone.features
        self.image_pool=nn.AdaptiveAvgPool2d(1)
        self.image_proj=nn.Sequential(nn.Flatten(),nn.Linear(backbone.classifier[0].in_features,64),nn.GELU())
        self.cgci=CGCIInspired(channels=6,meta_dim=16,out_dim=64,rank=8)
        self.tcn=TemporalConvNet(in_ch=6,channels=32,out_dim=64)
        self.meta_proj=nn.Sequential(nn.Linear(16,64),nn.GELU())
        self.bilinear=LowRankBilinear(64,64,64)
        self.fusion=HypergraphFusion(64)
        self.state=LowRankContinuousState(64,32,8)
        self.dropout=nn.Dropout(0.1)
        self.severity_head=nn.Linear(32,NUM_SEVERITY_CLASSES)
        self.risk_head=nn.Linear(32,len(HORIZONS))
        self.hazard_head=nn.Linear(32+1,4)  # four onset intervals; no-onset class is derived from survival
    def forward(self,image,sensor,sensor_desc,metadata,delta_days=None):
        v=self.image_pool(self.image_backbone(image))
        v=self.image_proj(v)
        c=self.cgci(sensor,sensor_desc,metadata)
        t=self.tcn(sensor)
        m=self.meta_proj(metadata)
        b=self.bilinear(v,t)
        nodes=torch.stack([v,t,c,m],dim=1)
        z=self.fusion(nodes) + b
        h,vel=self.state(z,delta_days)
        hdrop=self.dropout(h)
        sev=self.severity_head(hdrop)
        risk=self.risk_head(hdrop)
        hazard=self.hazard_head(torch.cat([hdrop,vel.norm(dim=1,keepdim=True)],dim=1))
        return {'severity_logits':sev,'risk_logits':risk,'hazard_logits':hazard,
                'latent':h,'velocity':vel}

model=SCHLightNet(pretrained=USE_PRETRAINED).to(DEVICE)
nparams=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {nparams:,}")

## 10. Losses, censored hazard likelihood, and training loop

In [ ]:
def discrete_hazard_nll(hazard_logits, onset_day, event_observed):
    # Four intervals: (0,7], (7,14], (14,21], (21,30]; hazard sigmoid per interval.
    hazards=torch.sigmoid(hazard_logits).clamp(1e-6,1-1e-6)
    batch=hazards.shape[0]
    losses=[]
    for i in range(batch):
        day=float(onset_day[i].detach().cpu())
        event=bool(float(event_observed[i].detach().cpu()) >= 0.5)
        if not event or day>30:
            # Right-censored / no observed onset within the prediction window: survive all intervals.
            losses.append(-torch.log1p(-hazards[i]).sum())
        else:
            k=0 if day<=7 else 1 if day<=14 else 2 if day<=21 else 3
            surv=-torch.log1p(-hazards[i,:k]).sum() if k>0 else hazards.new_tensor(0.)
            event_loss=-torch.log(hazards[i,k])
            losses.append(surv+event_loss)
    return torch.stack(losses).mean()

def batch_to_device(batch):
    return {k:(v.to(DEVICE) if torch.is_tensor(v) else v) for k,v in batch.items()}

def compute_loss(out,batch):
    sev_loss=F.cross_entropy(out['severity_logits'],batch['severity'])
    risk_loss=F.binary_cross_entropy_with_logits(out['risk_logits'],batch['risk'])
    haz_loss=discrete_hazard_nll(out['hazard_logits'],batch['onset_day'],batch['event_observed'])
    # Ordinal proximity penalty supplements categorical severity loss.
    probs=torch.softmax(out['severity_logits'],dim=1)
    expected=(probs*torch.arange(NUM_SEVERITY_CLASSES,device=DEVICE).float()).sum(1)
    ordinal_loss=F.smooth_l1_loss(expected,batch['severity'].float())
    total=sev_loss + risk_loss + 0.5*haz_loss + 0.25*ordinal_loss
    return total, {'severity':sev_loss.item(),'risk':risk_loss.item(),'hazard':haz_loss.item(),'ordinal':ordinal_loss.item()}

optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max(EPOCHS,1))
history=[]; best_val=float('inf'); best_path=OUTPUT_DIR/'sch_lightnet_best.pt'
patience=3; no_improve=0

for epoch in range(EPOCHS):
    model.train(); train_losses=[]
    for batch in tqdm(train_loader,desc=f"Epoch {epoch+1}/{EPOCHS}",leave=False):
        batch=batch_to_device(batch)
        optimizer.zero_grad(set_to_none=True)
        out=model(batch['image'],batch['sensor'],batch['sensor_desc'],batch['metadata'],batch['visit_day'])
        loss,parts=compute_loss(out,batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),2.0)
        optimizer.step()
        train_losses.append(loss.item())
    model.eval(); val_losses=[]
    with torch.no_grad():
        for batch in val_loader:
            batch=batch_to_device(batch)
            out=model(batch['image'],batch['sensor'],batch['sensor_desc'],batch['metadata'],batch['visit_day'])
            loss,_=compute_loss(out,batch); val_losses.append(loss.item())
    tr=float(np.mean(train_losses)) if train_losses else np.nan
    va=float(np.mean(val_losses)) if val_losses else tr
    history.append({'epoch':epoch+1,'train_loss':tr,'val_loss':va})
    print(f"Epoch {epoch+1:02d} | train={tr:.4f} | val={va:.4f}")
    if va < best_val:
        best_val=va; no_improve=0
        torch.save({'model_state_dict':model.state_dict(),'config':{'img_size':IMG_SIZE,'horizons':HORIZONS,
                   'severity_classes':NUM_SEVERITY_CLASSES,'data_mode':DATA_MODE}},best_path)
    else:
        no_improve+=1
        if no_improve>=patience: print('Early stopping'); break
    scheduler.step()

model.load_state_dict(torch.load(best_path,map_location=DEVICE)['model_state_dict'])
pd.DataFrame(history).to_csv(OUTPUT_DIR/'training_history.csv',index=False)
pd.DataFrame(history)

## 11. Evaluate detection, severity, risk, and hazard outputs

In [ ]:
@torch.no_grad()
def predict_loader(model,loader):
    model.eval(); rows=[]
    for batch in loader:
        batch=batch_to_device(batch)
        out=model(batch['image'],batch['sensor'],batch['sensor_desc'],batch['metadata'],batch['visit_day'])
        sev_prob=torch.softmax(out['severity_logits'],dim=1).cpu().numpy()
        risk_prob=torch.sigmoid(out['risk_logits']).cpu().numpy()
        haz_prob=torch.sigmoid(out['hazard_logits']).cpu().numpy()
        for i in range(len(sev_prob)):
            r={'episode_id':batch['episode_id'][i],
               'true_severity':int(batch['severity'][i].cpu()),
               'pred_severity':int(sev_prob[i].argmax()),
               'onset_class':int(batch['onset_class'][i].cpu()),
               'event_observed':float(batch['event_observed'][i].cpu()),
               'onset_day':float(batch['onset_day'][i].cpu())}
            for j,p in enumerate(sev_prob[i]): r[f'severity_prob_{j}']=float(p)
            for j,p in enumerate(risk_prob[i]): r[f'risk_prob_{HORIZONS[j]}d']=float(p)
            surv=1.0
            for j,p in enumerate(haz_prob[i]):
                r[f'hazard_{j+1}']=float(p)
                surv *= (1-float(p))
                r[f'cumulative_onset_prob_interval_{j+1}']=1-surv
            rows.append(r)
    return pd.DataFrame(rows)

test_pred=predict_loader(model,test_loader)
display(test_pred.head())

y_true=test_pred['true_severity'].to_numpy()
y_pred=test_pred['pred_severity'].to_numpy()
metrics={
 'severity_accuracy':accuracy_score(y_true,y_pred),
 'severity_macro_f1':f1_score(y_true,y_pred,average='macro',zero_division=0),
 'severity_weighted_f1':f1_score(y_true,y_pred,average='weighted',zero_division=0),
 'severity_quadratic_weighted_kappa':cohen_kappa_score(y_true,y_pred,weights='quadratic')
}
# Binary detection: any severity > 0 is sheath-blight-present. This is only valid if grade 0 means healthy.
yb=(y_true>0).astype(int); yp=(y_pred>0).astype(int)
metrics.update({
 'detection_accuracy':accuracy_score(yb,yp),
 'detection_precision':precision_score(yb,yp,zero_division=0),
 'detection_recall':recall_score(yb,yp,zero_division=0),
 'detection_f1':f1_score(yb,yp,zero_division=0)
})
for j,h in enumerate(HORIZONS):
    y_risk=np.array([r['risk'][j] for r in test_rows],dtype=int) if len(test_rows)==len(test_pred) else None
    p=test_pred[f'risk_prob_{h}d'].to_numpy()
    if y_risk is not None:
        metrics[f'risk_{h}d_brier']=brier_score_loss(y_risk,p)
        metrics[f'risk_{h}d_pr_auc']=average_precision_score(y_risk,p) if len(np.unique(y_risk))>1 else np.nan
        metrics[f'risk_{h}d_roc_auc']=roc_auc_score(y_risk,p) if len(np.unique(y_risk))>1 else np.nan

metrics_df=pd.DataFrame([metrics]).T.reset_index()
metrics_df.columns=['metric','value']
display(metrics_df)
metrics_df.to_csv(OUTPUT_DIR/'evaluation_metrics.csv',index=False)
test_pred.to_csv(OUTPUT_DIR/'test_predictions.csv',index=False)
print(classification_report(y_true,y_pred,zero_division=0))

## 12. Split-conformal onset prediction sets

In [ ]:
# Classification-style split conformal over the 5 onset classes:
# 0=(0,7], 1=(7,14], 2=(14,21], 3=(21,30], 4=no observed onset by day 30.
# This is a practical implementation; censoring-aware validity requires fully observed calibration labels
# and exchangeability, as described in the manuscript.
@torch.no_grad()
def collect_onset_class_probs(model, rows):
    loader=DataLoader(PaddyEpisodeDataset(rows),batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
    allp=[]; ally=[]; observed=[]
    model.eval()
    for b in loader:
        b=batch_to_device(b)
        out=model(b['image'],b['sensor'],b['sensor_desc'],b['metadata'],b['visit_day'])
        hz=torch.sigmoid(out['hazard_logits']).cpu().numpy()
        # interval masses and no-event mass from conditional hazards
        surv=np.ones(len(hz)); masses=[]
        for j in range(4):
            masses.append(surv*hz[:,j]); surv=surv*(1-hz[:,j])
        masses.append(surv)
        probs=np.stack(masses,axis=1)
        probs=probs/np.clip(probs.sum(axis=1,keepdims=True),1e-8,None)
        allp.extend(probs); ally.extend(b['onset_class'].cpu().numpy()); observed.extend(b['event_observed'].cpu().numpy())
    return np.asarray(allp),np.asarray(ally),np.asarray(observed)

cal_probs,cal_y,cal_obs=collect_onset_class_probs(model,calib_rows)
# Calibrate only on records with fully observed onset class. For a smoke test, include all classes as a code check.
valid=(cal_obs>=0.5) if np.any(cal_obs>=0.5) else np.ones_like(cal_obs,dtype=bool)
scores=1-cal_probs[np.arange(len(cal_y))[valid],cal_y[valid]]
alpha=0.10
if len(scores):
    q_level=min(1.0, math.ceil((len(scores)+1)*(1-alpha))/max(len(scores),1))
    qhat=float(np.quantile(scores,min(q_level,1.0),method='higher'))
else:
    qhat=1.0
print("Conformal threshold:",qhat,"calibration cases:",int(valid.sum()),"nominal coverage:",1-alpha)

test_probs,test_onset_y,test_obs=collect_onset_class_probs(model,test_rows)
prediction_sets=[]
for p in test_probs:
    included=np.where(1-p <= qhat)[0].tolist()
    if not included: included=[int(np.argmax(p))]
    prediction_sets.append(included)
coverage=np.mean([int(y in s) for y,s in zip(test_onset_y,prediction_sets)]) if len(test_onset_y) else np.nan
mean_width=np.mean([len(s) for s in prediction_sets]) if prediction_sets else np.nan
print(f"Empirical test coverage={coverage:.3f}; mean set size={mean_width:.2f}")
conformal_df=pd.DataFrame({'episode_id':[s['episode_id'] for s in test_rows],
                           'true_onset_class':test_onset_y,
                           'prediction_set':[','.join(map(str,s)) for s in prediction_sets]})
conformal_df.to_csv(OUTPUT_DIR/'conformal_prediction_sets.csv',index=False)

## 13. Plots (loss curves, confusion matrix, risk calibration)

In [ ]:
from sklearn.calibration import calibration_curve

hist_df=pd.DataFrame(history)
fig,ax=plt.subplots(figsize=(8,5))
ax.plot(hist_df.epoch,hist_df.train_loss,marker='o',label='Train loss')
ax.plot(hist_df.epoch,hist_df.val_loss,marker='s',label='Validation loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title('SCH-LightNet training history')
ax.grid(alpha=.3); ax.legend(); fig.tight_layout()
fig.savefig(OUTPUT_DIR/'training_curves.png',dpi=600,bbox_inches='tight'); plt.show()

cm=confusion_matrix(y_true,y_pred,labels=list(range(NUM_SEVERITY_CLASSES)))
fig,ax=plt.subplots(figsize=(8,7))
im=ax.imshow(cm,cmap='Blues')
ax.set_xticks(range(NUM_SEVERITY_CLASSES)); ax.set_yticks(range(NUM_SEVERITY_CLASSES))
ax.set_xlabel('Predicted severity'); ax.set_ylabel('True severity'); ax.set_title('Severity confusion matrix')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]): ax.text(j,i,str(cm[i,j]),ha='center',va='center')
fig.colorbar(im,ax=ax); fig.tight_layout()
fig.savefig(OUTPUT_DIR/'severity_confusion_matrix.png',dpi=600,bbox_inches='tight'); plt.show()

h=HORIZONS[1]
y_cal=np.array([s['risk'][1] for s in test_rows],dtype=int)
p_cal=test_pred[f'risk_prob_{h}d'].to_numpy()
if len(np.unique(y_cal))>1:
    frac_pos, mean_pred=calibration_curve(y_cal,p_cal,n_bins=5,strategy='quantile')
    fig,ax=plt.subplots(figsize=(6,6))
    ax.plot([0,1],[0,1],'k--',label='Perfect calibration')
    ax.plot(mean_pred,frac_pos,'o-',label=f'{h}-day risk')
    ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed fraction')
    ax.set_title('Risk calibration'); ax.grid(alpha=.3); ax.legend(); fig.tight_layout()
    fig.savefig(OUTPUT_DIR/'risk_calibration.png',dpi=600,bbox_inches='tight'); plt.show()
else:
    print('Calibration curve skipped: only one risk class in this test split.')

## 14. Grad-CAM visualization for the last image-backbone feature map

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model=model; self.target_layer=target_layer
        self.activations=None; self.gradients=None
        self.fwd=target_layer.register_forward_hook(self._save_activation)
        self.bwd=target_layer.register_full_backward_hook(self._save_gradient)
    def _save_activation(self,m,i,o): self.activations=o
    def _save_gradient(self,m,gi,go): self.gradients=go[0]
    def __call__(self, image, sensor, desc, meta, target_class=None):
        self.model.zero_grad(set_to_none=True)
        out=self.model(image,sensor,desc,meta)
        logits=out['severity_logits']
        cls=int(logits.argmax(1).item()) if target_class is None else int(target_class)
        logits[:,cls].sum().backward()
        weights=self.gradients.mean(dim=(2,3),keepdim=True)
        cam=(weights*self.activations).sum(dim=1,keepdim=True).relu()
        cam=F.interpolate(cam,size=(IMG_SIZE,IMG_SIZE),mode='bilinear',align_corners=False)
        cam=cam[0,0].detach().cpu().numpy()
        cam=(cam-cam.min())/(cam.max()-cam.min()+1e-8)
        return cam,cls
    def close(self): self.fwd.remove(); self.bwd.remove()

if len(test_rows):
    batch=next(iter(test_loader))
    img=batch['image'][0:1].to(DEVICE)
    sensor=batch['sensor'][0:1].to(DEVICE); desc=batch['sensor_desc'][0:1].to(DEVICE); meta=batch['metadata'][0:1].to(DEVICE)
    cam_tool=GradCAM(model,model.image_backbone[-1])
    cam,cls=cam_tool(img,sensor,desc,meta)
    cam_tool.close()
    # Convert normalized tensor back to display RGB
    mean=np.array([0.485,0.456,0.406]); std=np.array([0.229,0.224,0.225])
    base=img[0].detach().cpu().permute(1,2,0).numpy()*std+mean
    base=np.clip(base,0,1)
    heat=cv2.applyColorMap(np.uint8(255*cam),cv2.COLORMAP_JET)
    heat=cv2.cvtColor(heat,cv2.COLOR_BGR2RGB)/255.0
    overlay=np.clip(0.55*base+0.45*heat,0,1)
    fig,axs=plt.subplots(1,3,figsize=(14,4))
    axs[0].imshow(base); axs[0].set_title('Input image')
    axs[1].imshow(cam,cmap='jet'); axs[1].set_title(f'Grad-CAM (class {cls})')
    axs[2].imshow(overlay); axs[2].set_title('Overlay')
    for a in axs: a.axis('off')
    fig.tight_layout(); fig.savefig(OUTPUT_DIR/'gradcam_overlay.png',dpi=600,bbox_inches='tight'); plt.show()
else:
    print("No test rows available for Grad-CAM.")

## 15. Save model, preprocessing state, Excel workbook, and ZIP bundle

In [ ]:
import joblib
joblib.dump(scaler, OUTPUT_DIR/'sensor_robust_scaler.joblib')
with open(OUTPUT_DIR/'run_config.json','w') as f:
    json.dump({'data_mode':DATA_MODE,'seed':SEED,'img_size':IMG_SIZE,'sensor_channels':SENSOR_CHANNELS,
               'sensor_window':SENSOR_WINDOW,'horizons':HORIZONS,'epochs_requested':EPOCHS,
               'trainable_parameters':nparams,'conformal_alpha':alpha,'conformal_qhat':qhat},f,indent=2)

excel_path=OUTPUT_DIR/'SCH_LightNet_Results.xlsx'
with pd.ExcelWriter(excel_path,engine='openpyxl') as writer:
    pd.DataFrame(history).to_excel(writer,sheet_name='Training_History',index=False)
    metrics_df.to_excel(writer,sheet_name='Metrics',index=False)
    test_pred.to_excel(writer,sheet_name='Test_Predictions',index=False)
    conformal_df.to_excel(writer,sheet_name='Conformal_Sets',index=False)

zip_path=OUTPUT_DIR/'SCH_LightNet_Complete_Outputs.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.rglob('*'):
        if p.is_file() and p != zip_path:
            z.write(p,arcname=str(p.relative_to(OUTPUT_DIR)))
print("Saved outputs to:", OUTPUT_DIR)
print("Model:", best_path)
print("Excel:", excel_path)
print("ZIP:", zip_path)

## 16. Real-data checklist before reporting scientific results
1. Use the real image and sensor repositories named in the manuscript; keep licences and dataset attribution.
2. Build `episodes.csv` by matching image severity and sensor station/season episodes using the disclosed simulator-assisted procedure, or supply genuinely aligned field labels.
3. Ensure `episode_id` represents a field/plot-season unit and split by field and time (forward-chaining / leave-one-field-out), not randomly by individual visits.
4. Keep the conformal calibration set separate from gradient training. The notebook uses the validation partition as calibration and therefore does not use it for model selection beyond the simple loss trace; for publication, use a dedicated field-disjoint calibration set.
5. This reference implementation uses FFT summaries rather than a full synchrosqueezed wavelet transform, a differentiable low-rank continuous-time-inspired state update rather than a full natural-cubic-spline Neural CDE solver, and a compact hypergraph-style fusion block. Replace these approximations with the exact mathematical modules and report the differences if strict paper reproduction is required.
6. Do not cite smoke-test scores as model performance. Train/evaluate on field-level splits, repeat the five seeds, bootstrap confidence intervals, and benchmark latency/memory on the target edge devices.